In [1]:
import sys
import logging
from pathlib import Path
from typing import List, Mapping, Sequence, Union, Dict
import pandas as pd
import os

import pandas as pd
import dycomutils as common_utils

repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / "src").exists():
    repo_root = repo_root.parent

os.chdir(repo_root)
print(repo_root)
sys.path.append(str(repo_root))

from src.utils.graph_manager import GraphManager
from src.utils.utils import load_config, format_dataframe_to_string, regex_add_strings
from src.config.experiment import ExperimentConfig
from src.experiment.ground_truth import GT, SPARQLTemplate, GTAnswer
from evaluations.test_questions.templates import build_gt_from_template

CONFIG_PATH = "evaluations/calibration/config.yaml"
logging.info(f"Loading config: {CONFIG_PATH}")
lconfig = load_config(CONFIG_PATH)
config = ExperimentConfig.model_validate(lconfig)
config


/home/desild/work/research/LLM-Workflow-Explorer


ExperimentConfig(application=ApplicationInfo(description='The application that utilizes a functional approach to support unittesting of LLMs on user designed prompts. This application contains functions generated by LLMs and functions that utilizes LLM generated output within the system. In this experiments are represented as provone:Executions of provone:Programs (a step of the experiment pipeline) while some of these provone:Programs are generated by LLM as Generative_Tasks. dc:identifier represents the unique experiment id.'), file_paths=InputFiles(schema_loc='schema/schemaV2.json', execution_kg_loc='usecases/chatbs/data/1_sample_graph/chatbs_sample.ttl', metadata_loc='usecases/chatbs/data/1_sample_graph/chatbs_sample_metadata.json'), explorer_config=ExplorerConfig(kg_name='ChatBS-NexGen', ontology_triples_path='schema/extracted_ontology_triples.csv', parallel=True, temp_folder='evaluations/calibration/exeprog_creation/tmp/programs', use_cache=False, explorer_metadata_loc='evaluatio

In [2]:
graph_manager = GraphManager(
    config.ttl,
    config.file_paths.execution_kg_loc,
)

gt_list: List[GT] = []


In [3]:
# Question 1
question1 = """
How many unique \"experiment execution\" are there in this?
"""
question_sparql1 = """
SELECT (count(distinct ?ids) AS ?obj_count)
WHERE {
     ?obj a provone:Execution ;
          dcterms:identifier ?ids .
}
"""

entities1 = graph_manager.query(question_sparql1)
entities1


,obj_count
0,1


In [4]:
answer1 = "The answer to the question is 1 unique executions."

gt_list.append(
    GT(
        question=question1,
        answer=answer1,
        sparql_querys=[
            SPARQLTemplate(
                template=question_sparql1,
                description="This SPARQL query counts the number of unique executions by counting distinct identifiers in the provone:Execution class.",
            )
        ],
        qtype=["single", "numeric"],
        entities=entities1.to_dict(orient="records"),
    )
)


In [5]:
# Question 3
question3 = """
In what places do we utilize AI in this workflow?
"""
question_sparql3 = """
SELECT distinct ?obj ?desc
WHERE {
     ?obj a workflow:Generative_Task .
     ?obj dc:description ?desc .
}
"""

entities3 = graph_manager.query(question_sparql3, resolve_curie=True)
entities3


,obj,desc
0,ChatBS-NexGen:Generative_Task-id_2026041220074...,LLM Generates an text for \nsystem prompt: You...
1,ChatBS-NexGen:Generative_Task-id_2026041220074...,Extracts important entities for \ntext: Before...
2,ChatBS-NexGen:Generative_Task-information_extr...,Generates the Information Extractor Function u...
3,ChatBS-NexGen:Generative_Task-query_result_pos...,Generates the post processing Function using L...
4,ChatBS-NexGen:Generative_Task-system_prompt_ge...,Generates the System Prompt Template using Lar...


In [6]:
answer3 = """
The ChatBS System utilizes LLMs to generate the following functions

1. Information extraction function
2. System Prompt template function
3. Sparql result post processing function

and further it's used within functions to
1. Generate the LLM outputs,
2. Used to extract key information from the system.
"""

gt_list.append(
    GT(
        question=question3,
        answer=answer3,
        sparql_querys=[SPARQLTemplate(template=question_sparql3, description="")],
        entities=entities3.to_dict(orient="records"),
        qtype=["multi", "entity"],
    )
)


In [7]:
# Question 4
question4 = """
What are the extracted KG entities for the execution with id 1_1
"""
question_sparql4 = """
SELECT DISTINCT ?member ?prop ?value
WHERE {
     ?obj dc:description ?desc .
     FILTER(REGEX(?desc, "SPARQL queries to extract information", "i"))

     ?obj a provone:Program .
     ?exe prov:qualifiedAssociation/prov:hadPlan ?obj .
     ?exe dcterms:identifier ?id .
     FILTER(REGEX(?id, "1_1", "i"))

     ?data prov:wasGeneratedBy ?exe .
     ?data a ?class .

     {
         ?data a provone:Collection .
         ?data provone:hadMember ?member .
         ?member ?prop ?value .
     }
     UNION
     {
         ?data a provone:Data .
         BIND(?data AS ?member)
         ?data ?prop ?value .
     }
}
"""

entities4 = graph_manager.query(question_sparql4, resolve_curie=True)
entities4


,member,prop,value
0,ChatBS-NexGen:Data-id_20260412200743_406-kg_info,DFColumn:sugarG,NA@en^^<xsd:string>
1,ChatBS-NexGen:Data-id_20260412200743_406-kg_info,rdfs:label,row_1@en^^<xsd:string>
2,ChatBS-NexGen:Data-id_20260412200743_720-kg_info,DFColumn:sugarG,@en^^<xsd:string>
3,ChatBS-NexGen:Data-id_20260412200743_720-kg_info,DFColumn:ingredientCat,@en^^<xsd:string>
4,ChatBS-NexGen:Data-id_20260412200743_406-kg_info,DFColumn:ingredientCat,NA@en^^<xsd:string>
...,...,...,...
168,ChatBS-NexGen:Data-id_20260412200742_324-extracts,prov:wasGeneratedBy,ChatBS-NexGen:id_20260412200745_473
169,ChatBS-NexGen:Data-id_20260412200742_324-extracts,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,provone:Data
170,ChatBS-NexGen:Data-id_20260412200742_324-extracts,rdfs:label,extracts@en^^<xsd:string>
171,ChatBS-NexGen:Data-id_20260412200742_324-extracts,prov:value,Lettuce@en^^<xsd:string>


In [8]:
pentities4 = set(
    entities4.loc[entities4["prop"] == "DFColumn:ingredientName", "value"]
    .apply(lambda x: x.split("@")[0])
    .tolist()
) - {"", "NA"}

answer4 = """
The retrieved entities from the knowledge graph includes the following ingredients,

1. Yogurt
2. Berry
3. Cheese
4. Carrot
5. Turkey
6. Bell pepper
7. Hummus
8. Almond
9. Avocado
10. Pumpkin seed
11. Water
12. Pineapple
13. Cottage cheese
"""

gt_list.append(
    GT(
        question=question4,
        answer=answer4,
        sparql_querys=[SPARQLTemplate(template=question_sparql4, description="")],
        entities=list(pentities4),
        qtype=["multi", "literal"]
    )
)


In [9]:
# Question 5
question5 = """
why did some ingredients have missing values for `sugarG` and `ingredientCat` columns in the final output of the workflow?
"""
question_sparql5 = """
SELECT DISTINCT ?member ?prop ?value
WHERE {
     ?obj dc:description ?desc .
     FILTER(REGEX(?desc, "SPARQL queries to extract information", "i"))

     ?obj a provone:Program .
     ?exe prov:qualifiedAssociation/prov:hadPlan ?obj .
     ?exe dcterms:identifier ?id .
     FILTER(REGEX(?id, "1_1", "i"))

     ?data prov:wasGeneratedBy ?exe .
     ?data a ?class .

     {
         ?data a provone:Collection .
         ?data provone:hadMember ?member .
         ?member ?prop ?value .
     }
     UNION
     {
         ?data a provone:Data .
         BIND(?data AS ?member)
         ?data ?prop ?value .
     }
}

order by ?member ?prop
"""

entities5 = graph_manager.query(question_sparql5, resolve_curie=True)
entities5


,member,prop,value
0,ChatBS-NexGen:Data-id_20260412200742_128-extracts,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,provone:Data
1,ChatBS-NexGen:Data-id_20260412200742_128-extracts,rdfs:label,extracts@en^^<xsd:string>
2,ChatBS-NexGen:Data-id_20260412200742_128-extracts,prov:value,Berry@en^^<xsd:string>
3,ChatBS-NexGen:Data-id_20260412200742_128-extracts,prov:wasGeneratedBy,ChatBS-NexGen:id_20260412200743_952
4,ChatBS-NexGen:Data-id_20260412200742_128-extracts,prov:wasGeneratedBy,ChatBS-NexGen:id_20260412200742_397
...,...,...,...
168,ChatBS-NexGen:Data-id_20260412200745_516-kg_info,DFColumn:ingredientName,@en^^<xsd:string>
169,ChatBS-NexGen:Data-id_20260412200745_516-kg_info,DFColumn:sugarG,@en^^<xsd:string>
170,ChatBS-NexGen:Data-id_20260412200745_516-kg_info,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,provone:Data
171,ChatBS-NexGen:Data-id_20260412200745_516-kg_info,rdfs:label,row_0@en^^<xsd:string>


In [10]:
na_entities = entities5.loc[
    entities5["value"].isin(["@en^^<xsd:string>", "NA@en^^<xsd:string>"]),
    "member",
].unique().tolist()
pentities5 = set(
    entities5.loc[
        entities5["member"].isin(na_entities)
        & (entities5["prop"] == "DFColumn:ingredientName"),
        "value",
    ]
    .apply(lambda x: x.split("@")[0])
    .tolist()
) - {"", "NA"}

answer5 = """
Due to the fact that some information is not available in the knowledge graph for ingredients such as
'Turkey', 'Hummus', 'Cheese', 'Cottage cheese', 'Berry', 'Water' the sugarG and ingredientCat is not available.
"""

gt_list.append(
    GT(
        question=question5,
        answer=answer5,
        sparql_querys=[SPARQLTemplate(template=question_sparql5, description="")],
        entities=list(pentities5),
        qtype=["multi", "literal"]
    )
)


In [11]:
# Question MULT 1: instructions for function generation.


template_spec = [{
    "func_desc":"sparql query post processing"
}, {
    "func_desc": "system prompt template"
}, {
    "func_desc": "information extraction"
}]

sparql_spec = [{
    "func_desc_term":"Post-processes the query results"
},{
    "func_desc_term":"Generates a system prompt"
},{
    "func_desc_term":"Extracts entities or relevant information"
},]

question_template = """
what are the instructions used by the LLM to generate the {func_desc} function step in the pipeline?
"""
sparql_template = """
SELECT distinct ?obj ?llm ?inpd
WHERE {
     ?obj dc:description ?desc .
     ?obj a ?class .
     FILTER(REGEX(?desc, "{func_desc_term}","i"))

     ?llm_out sio:SIO_000202 ?obj .
     ?llm_out sio:SIO_000232 ?llm .
     ?llm sio:SIO_000230 ?inp .
     ?inp prov:value ?inpd
}
"""

def answer_template_function(
    entities:pd.DataFrame, 
    question_specs:Mapping[str, Union[str,int]], 
    sparql_specs:Mapping[str, Union[str,int]]
    ) -> GTAnswer:

    answer_template = """
    We utilize the following input instructions to generate the function responsible for {func_desc}.

    {answer_inst}
    """
    
    
    _ent = entities[['obj', 'inpd']]
    _temp_ent = format_dataframe_to_string(
        _ent,
        header_map={
            "obj": "Funtion",
            "inpd": "Instruction for function generation"
        }
    )
    
    answer_nlp = regex_add_strings(
        answer_template, 
        func_desc = question_specs["func_desc"],
        answer_inst = _temp_ent
        )
    
    return GTAnswer(
        answer_nlp= answer_nlp,
        entities= _ent.to_dict(orient="records")
    )

gts = build_gt_from_template(
        template=question_template,
        answer_template=answer_template_function,
        sparql_query=sparql_template,
        specs_template=template_spec,
        specs_sparql=sparql_spec,
        graph_manager= graph_manager,
        verbose=True
    )

for gt in gts:
    gt.qtype = ["multi", "entity", "literal"]
gt_list.extend(gts)

ic| rquestion: '''
                what are the instructions used by the LLM to generate the sparql query post processing function step in the pipeline?
                '''
ic| entities:                                                  obj  \
              0  http://testwebsite/testProgram#query_result_po...   
              1  http://testwebsite/testProgram#query_result_po...   
              
                                                               llm  \
              0  http://testwebsite/testProgram#LLM-query_resul...   
              1  http://testwebsite/testProgram#LLM-query_resul...   
              
                                                              inpd  
              0  Concat all the rows into single row at each co...  
              1  If there are duplicate ingredients then select...  
ic| answer: GTAnswer(answer_nlp="
                We utilize the following input instructions to generate the function responsible for sparql query post processing.
     

In [12]:
# Question MULT 2: input ports of the functions


template_spec = [{
    "func_desc":"sparql query post processing"
}, {
    "func_desc": "system prompt template"
}, {
    "func_desc": "information extraction"
}, {
    "func_desc": "sparql query function"
}, {
    "func_desc": "llm chat function"
}]

sparql_spec = [{
    "func_desc_term":"Post-processes the query results"
},{
    "func_desc_term":"Generates a system prompt"
},{
    "func_desc_term":"Extracts entities or relevant information"
}, {
    "func_desc_term": "Builds SPARQL"
}, {
    "func_desc_term": "Chat with"
}]

question_template = """
what are the input parameters of the function used in {func_desc} step in the pipeline?
"""
sparql_template = """
SELECT distinct ?flbl ?lbl
WHERE {
     ?obj dc:description ?desc .
     ?obj rdfs:label ?flbl . 
     ?obj a ?class .
     FILTER(REGEX(?desc, "{func_desc_term}","i"))

     ?obj provone:hasInPort ?port .
     ?port rdfs:label ?lbl .
}
"""

def answer_template_function(
    entities:pd.DataFrame, 
    question_specs:Mapping[str, Union[str,int]], 
    sparql_specs:Mapping[str, Union[str,int]]
    ) -> GTAnswer:

    answer_template = """
    The input parameters to {func_desc} function.

    {answer_inst}
    """
    
    _ent = entities[['flbl', 'lbl']]
    _temp_ent = format_dataframe_to_string(
        _ent,
        header_map={
            "flbl": "Funtion name",
            "lbl": "Function returns"
        }
    )
    
    answer_nlp = regex_add_strings(
        answer_template, 
        func_desc = question_specs["func_desc"],
        answer_inst = _temp_ent
        )
    
    return GTAnswer(
        answer_nlp= answer_nlp,
        entities= _ent.to_dict(orient="records")
    )

gts = build_gt_from_template(
        template=question_template,
        answer_template=answer_template_function,
        sparql_query=sparql_template,
        specs_template=template_spec,
        specs_sparql=sparql_spec,
        graph_manager= graph_manager,
        verbose=True
    )


for gt in gts:
    gt.qtype = ["multi", "literal"]
gt_list.extend(gts)

ic| rquestion: '''
                what are the input parameters of the function used in sparql query post processing step in the pipeline?
                '''
ic| entities:                                            flbl  \
              0  Query Result Post-Processor@en^^<xsd:string>   
              
                                                               lbl  
              0  Post-processing input port: kg_knowledge@en^^<...  
ic| answer: GTAnswer(answer_nlp='
                The input parameters to sparql query post processing function.
            
                Funtion name | Function returns
            Query Result Post-Processor@en^^<xsd:string> | Post-processing input port: kg_knowledge@en^^<xsd:string>
                ', entities=[{'flbl': 'Query Result Post-Processor@en^^<xsd:string>', 'lbl': 'Post-processing input port: kg_knowledge@en^^<xsd:string>'}])
ic| rquestion: '''
                what are the input parameters of the function used in system prompt templat

In [13]:
# Question MULT 3 : output ports of functions


template_spec = [{
    "func_desc":"sparql query post processing"
}, {
    "func_desc": "system prompt template"
}, {
    "func_desc": "information extraction"
}, {
    "func_desc": "sparql query function"
}, {
    "func_desc": "llm chat function"
}]

sparql_spec = [{
    "func_desc_term":"Post-processes the query results"
},{
    "func_desc_term":"Generates a system prompt"
},{
    "func_desc_term":"Extracts entities or relevant information"
}, {
    "func_desc_term": "Builds SPARQL queries to extract"
}, {
    "func_desc_term": "Chat with the LLM"
}]

question_template = """
what are the input parameters of the function used in {func_desc} step in the pipeline?
"""
sparql_template = """
SELECT distinct ?flbl ?lbl
WHERE {
     ?obj dc:description ?desc .
     ?obj rdfs:label ?flbl . 
     ?obj a ?class .
     FILTER(REGEX(?desc, "{func_desc_term}","i"))

     ?obj provone:hasOutPort ?port .
     ?port rdfs:label ?lbl .
}
"""

def answer_template_function(
    entities:pd.DataFrame, 
    question_specs:Mapping[str, Union[str,int]], 
    sparql_specs:Mapping[str, Union[str,int]]
    ) -> GTAnswer:

    answer_template = """
    The input parameters to {func_desc} function.

    {answer_inst}
    """
    
    _ent = entities[['flbl', 'lbl']]
    _temp_ent = format_dataframe_to_string(
        _ent,
        header_map={
            "flbl": "Funtion name",
            "lbl": "Function returns"
        }
    )
    
    answer_nlp = regex_add_strings(
        answer_template, 
        func_desc = question_specs["func_desc"],
        answer_inst = _temp_ent
        )
    
    return GTAnswer(
        answer_nlp= answer_nlp,
        entities= _ent.to_dict(orient="records")
    )

gts = build_gt_from_template(
        template=question_template,
        answer_template=answer_template_function,
        sparql_query=sparql_template,
        specs_template=template_spec,
        specs_sparql=sparql_spec,
        graph_manager= graph_manager,
        verbose=True
    )

for gt in gts:
    gt.qtype = ["multi", "literal"]
gt_list.extend(gts)

ic| rquestion: '''
                what are the input parameters of the function used in sparql query post processing step in the pipeline?
                '''
ic| entities:                                            flbl  \
              0  Query Result Post-Processor@en^^<xsd:string>   
              
                                                               lbl  
              0  Post-processed output port: post_processed_out...  
ic| answer: GTAnswer(answer_nlp='
                The input parameters to sparql query post processing function.
            
                Funtion name | Function returns
            Query Result Post-Processor@en^^<xsd:string> | Post-processed output port: post_processed_output@en^^<xsd:string>
                ', entities=[{'flbl': 'Query Result Post-Processor@en^^<xsd:string>', 'lbl': 'Post-processed output port: post_processed_output@en^^<xsd:string>'}])
ic| rquestion: '''
                what are the input parameters of the function used in sys

In [14]:
# Question MULT 4 : connected functions


template_spec = [{
    "func_desc":"sparql query post processing"
}, {
    "func_desc": "system prompt template"
}, {
    "func_desc": "information extraction"
}, {
    "func_desc": "sparql query function"
}, {
    "func_desc": "llm chat function"
}]

sparql_spec = [{
    "func_desc_term":"Post-processes the query results"
},{
    "func_desc_term":"Generates a system prompt"
},{
    "func_desc_term":"Extracts entities or relevant information"
}, {
    "func_desc_term": "Builds SPARQL queries to extract"
}, {
    "func_desc_term": "Chat with the LLM"
}]

question_template = """
what are the connected functions to the function used in {func_desc} step in the pipeline?
"""
sparql_template = """

SELECT DISTINCT ?fn2 ?fn2lbl
WHERE {
    ?obj dc:description ?desc .
    ?obj rdfs:label ?flbl . 
    ?obj a ?class .
    FILTER(REGEX(?desc, "{func_desc_term}","i"))

    ?obj (provone:hasInPort | provone:hasOutPort) ?port .
    ?port provone:connectsTo ?channel .
    ?portt provone:connectsTo ?channel .
    ?fn2 (provone:hasInPort | provone:hasOutPort) ?portt .
    FILTER(?obj != ?fn2)
    
    ?fn2 rdfs:label ?fn2lbl .
}
"""

def answer_template_function(
    entities:pd.DataFrame, 
    question_specs:Mapping[str, Union[str,int]], 
    sparql_specs:Mapping[str, Union[str,int]]
    ) -> GTAnswer:

    answer_template = """
    The connected functions to {func_desc} function.

    {answer_inst}
    """
    
    _ent = entities[['fn2', 'fn2lbl']]
    _temp_ent = format_dataframe_to_string(
        _ent,
        header_map={
            "fn2": "Funtion URI",
            "fn2lbl": "Function name"
        }
    )
    
    answer_nlp = regex_add_strings(
        answer_template, 
        func_desc = question_specs["func_desc"],
        answer_inst = _temp_ent
        )
    
    return GTAnswer(
        answer_nlp= answer_nlp,
        entities= _ent.to_dict(orient="records")
    )

gts = build_gt_from_template(
        template=question_template,
        answer_template=answer_template_function,
        sparql_query=sparql_template,
        specs_template=template_spec,
        specs_sparql=sparql_spec,
        graph_manager= graph_manager,
        verbose=True
    )

for gt in gts:
    gt.qtype = ["multi", "literal", "entity"]
gt_list.extend(gts)

ic| rquestion: '''
                what are the connected functions to the function used in sparql query post processing step in the pipeline?
                '''
ic| entities:                                                  fn2  \
              0  http://testwebsite/testProgram#batch_sparql_qu...   
              
                                                        fn2lbl  
              0  Batch SPARQL Query Extractor@en^^<xsd:string>  
ic| answer: GTAnswer(answer_nlp='
                The connected functions to sparql query post processing function.
            
                Funtion URI | Function name
            http://testwebsite/testProgram#batch_sparql_query_extractor | Batch SPARQL Query Extractor@en^^<xsd:string>
                ', entities=[{'fn2': 'http://testwebsite/testProgram#batch_sparql_query_extractor', 'fn2lbl': 'Batch SPARQL Query Extractor@en^^<xsd:string>'}])
ic| rquestion: '''
                what are the connected functions to the function used in system

In [15]:
# Question MULT 5 : number of execution of the functions


template_spec = [{
    "func_desc":"sparql query post processing"
}, {
    "func_desc": "system prompt template"
}, {
    "func_desc": "information extraction"
}, {
    "func_desc": "sparql query function"
}, {
    "func_desc": "llm chat function"
}]

sparql_spec = [{
    "func_desc_term":"Post-processes the query results"
},{
    "func_desc_term":"Generates a system prompt"
},{
    "func_desc_term":"Extracts entities or relevant information"
}, {
    "func_desc_term": "Builds SPARQL queries to extract"
}, {
    "func_desc_term": "Chat with the LLM"
}]

question_template = """
what are the executions of the {func_desc} step in the pipeline?
"""
sparql_template = """

SELECT DISTINCT ?exe
WHERE {
    ?obj dc:description ?desc .
    ?obj rdfs:label ?flbl . 
    ?obj a ?class .
    FILTER(REGEX(?desc, "{func_desc_term}","i"))

    ?asso prov:hadPlan ?obj .
    ?exe prov:qualifiedAssociation ?asso .
}
"""

def answer_template_function(
    entities:pd.DataFrame, 
    question_specs:Mapping[str, Union[str,int]], 
    sparql_specs:Mapping[str, Union[str,int]]
    ) -> GTAnswer:

    answer_template = """
    The executions of the  {func_desc} function are.

    {answer_inst}
    """
    
    _ent = entities[['exe']]
    _temp_ent = format_dataframe_to_string(
        _ent,
        header_map={
            "exe": "Executions"
        }
    )
    
    answer_nlp = regex_add_strings(
        answer_template, 
        func_desc = question_specs["func_desc"],
        answer_inst = _temp_ent
        )
    
    return GTAnswer(
        answer_nlp= answer_nlp,
        entities= _ent.to_dict(orient="records")
    )

gts = build_gt_from_template(
        template=question_template,
        answer_template=answer_template_function,
        sparql_query=sparql_template,
        specs_template=template_spec,
        specs_sparql=sparql_spec,
        graph_manager= graph_manager,
        verbose=True
    )

for gt in gts:
    gt.qtype = ["multi", "entity"]
gt_list.extend(gts)

ic| rquestion: '''
                what are the executions of the sparql query post processing step in the pipeline?
                '''
ic| entities:                                                  exe
              0  http://testwebsite/testProgram#id_202604122007...
ic| answer: GTAnswer(answer_nlp='
                The executions of the  sparql query post processing function are.
            
                Executions
            http://testwebsite/testProgram#id_20260412200745_507
                ', entities=[{'exe': 'http://testwebsite/testProgram#id_20260412200745_507'}])
ic| rquestion: '''
                what are the executions of the system prompt template step in the pipeline?
                '''
ic| entities:                                                  exe
              0  http://testwebsite/testProgram#id_202604122007...
ic| answer: GTAnswer(answer_nlp='
                The executions of the  system prompt template function are.
            
                Execution

In [16]:
# Question MULT 6 : who was the agent that ran the functions 


template_spec = [{
    "func_desc":"sparql query post processing"
}, {
    "func_desc": "system prompt template"
}, {
    "func_desc": "information extraction"
}, {
    "func_desc": "sparql query function"
}, {
    "func_desc": "llm chat function"
}]

sparql_spec = [{
    "func_desc_term":"Post-processes the query results"
},{
    "func_desc_term":"Generates a system prompt"
},{
    "func_desc_term":"Extracts entities or relevant information"
}, {
    "func_desc_term": "Builds SPARQL queries to extract"
}, {
    "func_desc_term": "Chat with the LLM"
}]

question_template = """
who were the agents that ran the {func_desc} step in the pipeline?
"""
sparql_template = """

SELECT DISTINCT ?agent ?albl
WHERE {
    ?obj dc:description ?desc .
    ?obj rdfs:label ?flbl . 
    ?obj a ?class .
    FILTER(REGEX(?desc, "{func_desc_term}","i"))

    ?asso prov:hadPlan ?obj .
    ?asso prov:agent ?agent .
    ?agent rdfs:label ?albl
}
"""

def answer_template_function(
    entities:pd.DataFrame, 
    question_specs:Mapping[str, Union[str,int]], 
    sparql_specs:Mapping[str, Union[str,int]]
    ) -> GTAnswer:

    answer_template = """
    The executions of the  {func_desc} function are.

    {answer_inst}
    """
    
    _ent = entities[['albl']]
    _temp_ent = format_dataframe_to_string(
        _ent,
        header_map={
            "albl": "Agents"
        }
    )
    
    answer_nlp = regex_add_strings(
        answer_template, 
        func_desc = question_specs["func_desc"],
        answer_inst = _temp_ent
        )
    
    return GTAnswer(
        answer_nlp= answer_nlp,
        entities= _ent.to_dict(orient="records")
    )

gts = build_gt_from_template(
        template=question_template,
        answer_template=answer_template_function,
        sparql_query=sparql_template,
        specs_template=template_spec,
        specs_sparql=sparql_spec,
        graph_manager= graph_manager,
        verbose=True
    )

for gt in gts:
    gt.qtype = ["multi", "entity", "literal"]
gt_list.extend(gts)

ic| rquestion: '''
                who were the agents that ran the sparql query post processing step in the pipeline?
                '''
ic| entities:                                       agent       albl
              0  http://testwebsite/testProgram#test_user  Test User
ic| answer: GTAnswer(answer_nlp='
                The executions of the  sparql query post processing function are.
            
                Agents
            Test User
                ', entities=[{'albl': 'Test User'}])
ic| rquestion: '''
                who were the agents that ran the system prompt template step in the pipeline?
                '''
ic| entities:                                       agent       albl
              0  http://testwebsite/testProgram#test_user  Test User
ic| answer: GTAnswer(answer_nlp='
                The executions of the  system prompt template function are.
            
                Agents
            Test User
                ', entities=[{'albl': 'Test User'}])
ic| 

In [17]:
# Question MULT 7 : Is the function made by a Generative Task  


template_spec = [{
    "func_desc":"sparql query post processing"
}, {
    "func_desc": "system prompt template"
}, {
    "func_desc": "information extraction"
}, {
    "func_desc": "sparql query function"
}, {
    "func_desc": "llm chat function"
}]

sparql_spec = [{
    "func_desc_term":"Post-processes the query results"
},{
    "func_desc_term":"Generates a system prompt"
},{
    "func_desc_term":"Extracts entities or relevant information"
}, {
    "func_desc_term": "Builds SPARQL queries to extract"
}, {
    "func_desc_term": "Chat with the LLM"
}]

question_template = """
Is the {func_desc} function made by a Generative Task using AI?
"""

sparql_template = """
SELECT DISTINCT ?fnlbl ?llm_gen_out
WHERE {
    ?obj dc:description ?desc .
    ?obj rdfs:label ?fnlbl .
    FILTER(REGEX(?desc, "{func_desc_term}", "i"))

    OPTIONAL {
        ?llm_out sio:SIO_000202 ?obj .
        ?llm_out sio:SIO_000232 ?llm .
        ?llm_gen prov:used ?llm .
    }

    BIND(COALESCE(?llm_gen, "None") AS ?llm_gen_out)
}
"""

def answer_template_function(
    entities:pd.DataFrame, 
    question_specs:Mapping[str, Union[str,int]], 
    sparql_specs:Mapping[str, Union[str,int]]
    ) -> GTAnswer:

    answer_template = """
    The {func_desc} function {decision} by a Generative Task.

    {answer_inst}
    """
    
    _ent = entities[['fnlbl', 'llm_gen_out']]
    _temp_ent = format_dataframe_to_string(
        _ent,
        header_map={
            "obj": "Function",
            'llm_gen_out': "Generative Task used to generate function"
        }
    )
    
    answer_nlp = regex_add_strings(
        answer_template, 
        func_desc = question_specs["func_desc"],
        answer_inst = _temp_ent,
        decision = "is made" if _ent['llm_gen_out'].tolist()[0] != "None" else "is not made"
        )
    
    return GTAnswer(
        answer_nlp= answer_nlp,
        entities= _ent.to_dict(orient="records")
    )

gts = build_gt_from_template(
        template=question_template,
        answer_template=answer_template_function,
        sparql_query=sparql_template,
        specs_template=template_spec,
        specs_sparql=sparql_spec,
        graph_manager= graph_manager,
        verbose=True
    )

for gt in gts:
    gt.qtype = ["single", "bool"]
    gt.decision = gt.entities[0]['llm_gen_out'] != "None"
gt_list.extend(gts)

ic| rquestion: '''
                Is the sparql query post processing function made by a Generative Task using AI?
                '''
ic| entities:                                           fnlbl  \
              0  Query Result Post-Processor@en^^<xsd:string>   
              
                                                       llm_gen_out  
              0  http://testwebsite/testProgram#Generative_Task...  
ic| answer: GTAnswer(answer_nlp='
                The sparql query post processing function is made by a Generative Task.
            
                fnlbl | Generative Task used to generate function
            Query Result Post-Processor@en^^<xsd:string> | http://testwebsite/testProgram#Generative_Task-query_result_post_processor
                ', entities=[{'fnlbl': 'Query Result Post-Processor@en^^<xsd:string>', 'llm_gen_out': 'http://testwebsite/testProgram#Generative_Task-query_result_post_processor'}])
ic| rquestion: '''
                Is the system prompt template

ic| rquestion: '''
                Is the sparql query function function made by a Generative Task using AI?
                '''
ic| entities:                                    fnlbl llm_gen_out
              0  SPARQL Query Builder@en^^<xsd:string>        None
ic| answer: GTAnswer(answer_nlp='
                The sparql query function function is not made by a Generative Task.
            
                fnlbl | Generative Task used to generate function
            SPARQL Query Builder@en^^<xsd:string> | None
                ', entities=[{'fnlbl': 'SPARQL Query Builder@en^^<xsd:string>', 'llm_gen_out': 'None'}])
ic| rquestion: '''
                Is the llm chat function function made by a Generative Task using AI?
                '''
ic| entities:                        fnlbl llm_gen_out
              0  LLM Chat@en^^<xsd:string>        None
ic| answer: GTAnswer(answer_nlp='
                The llm chat function function is not made by a Generative Task.
            
              

In [18]:
# Question MULT 8 : Does any of the executions of the function contain a Generative Task  


template_spec = [{
    "func_desc":"sparql query post processing"
}, {
    "func_desc": "system prompt template"
}, {
    "func_desc": "information extraction"
}, {
    "func_desc": "sparql query function"
}, {
    "func_desc": "llm chat function"
}]

sparql_spec = [{
    "func_desc_term":"Post-processes the query results"
},{
    "func_desc_term":"Generates a system prompt"
},{
    "func_desc_term":"Extracts entities or relevant information"
}, {
    "func_desc_term": "Builds SPARQL queries to extract"
}, {
    "func_desc_term": "Chat with the LLM"
}]

question_template = """
Does any of the executions of the {func_desc} function made has a Generative Task as a component within?
"""

sparql_template = """
SELECT DISTINCT ?flbl ?exe ?llm_gen_out
WHERE {
    ?obj dc:description ?desc .
    ?obj rdfs:label ?flbl .
    FILTER(REGEX(?desc, "{func_desc_term}","i"))

    ?asso prov:hadPlan ?obj .
    ?exe prov:qualifiedAssociation ?asso .

    OPTIONAL {
        ?exe sio:SIO_000369 ?llm_gen .
    }

    BIND(COALESCE(?llm_gen, "None") AS ?llm_gen_out)
}
"""

def answer_template_function(
    entities:pd.DataFrame, 
    question_specs:Mapping[str, Union[str,int]], 
    sparql_specs:Mapping[str, Union[str,int]]
    ) -> GTAnswer:

    answer_template = """
    Executions of the {func_desc} function {decision} a Generative Task that utilize an LLM.

    {answer_inst}
    """
    
    _ent = entities[['flbl', 'exe', 'llm_gen_out']]
    _temp_ent = format_dataframe_to_string(
        _ent,
        header_map={
            "flbl": "Function",
            'exe': "Execution",
            'llm_gen_out': "Generative Task used within execution"
        }
    )
    
    answer_nlp = regex_add_strings(
        answer_template, 
        func_desc = question_specs["func_desc"],
        answer_inst = _temp_ent,
        decision = "contains" if _ent['llm_gen_out'].tolist()[0] != "None" else "does not contain"
        )
    
    return GTAnswer(
        answer_nlp= answer_nlp,
        entities= _ent.to_dict(orient="records")
    )

gts = build_gt_from_template(
        template=question_template,
        answer_template=answer_template_function,
        sparql_query=sparql_template,
        specs_template=template_spec,
        specs_sparql=sparql_spec,
        graph_manager= graph_manager,
        verbose=True
    )

for gt in gts:
    gt.qtype = ["single", "bool"]
    gt.decision = gt.entities[0]['llm_gen_out'] != "None"
gt_list.extend(gts)

ic| rquestion: '''
                Does any of the executions of the sparql query post processing function made has a Generative Task as a component within?
                '''
ic| entities:                                            flbl  \
              0  Query Result Post-Processor@en^^<xsd:string>   
              
                                                               exe llm_gen_out  
              0  http://testwebsite/testProgram#id_202604122007...        None  
ic| answer: GTAnswer(answer_nlp='
                Executions of the sparql query post processing function does not contain a Generative Task that utilize an LLM.
            
                Function | Execution | Generative Task used within execution
            Query Result Post-Processor@en^^<xsd:string> | http://testwebsite/testProgram#id_20260412200745_507 | None
                ', entities=[{'flbl': 'Query Result Post-Processor@en^^<xsd:string>', 'exe': 'http://testwebsite/testProgram#id_20260412200745_50

In [19]:
# Question MULT 9 : Does any of the executions of the function contain a Generative Task within and the function itself is generated by an LLM


template_spec = [{
    "func_desc":"sparql query post processing"
}, {
    "func_desc": "system prompt template"
}, {
    "func_desc": "information extraction"
}, {
    "func_desc": "sparql query function"
}, {
    "func_desc": "llm chat function"
}]

sparql_spec = [{
    "func_desc_term":"Post-processes the query results"
},{
    "func_desc_term":"Generates a system prompt"
},{
    "func_desc_term":"Extracts entities or relevant information"
}, {
    "func_desc_term": "Builds SPARQL queries to extract"
}, {
    "func_desc_term": "Chat with the LLM"
}]

question_template = """
Does any of the executions of the {func_desc} function made has a Generative Task within as a component and also {func_desc} 
function itself is generated by a Generative Task?
"""

sparql_template = """
SELECT DISTINCT ?flbl ?exe ?llm_gen_out ?llm_gen_out2
WHERE {
    ?obj dc:description ?desc .
    ?obj rdfs:label ?flbl .
    FILTER(REGEX(?desc, "{func_desc_term}","i"))

    ?asso prov:hadPlan ?obj .
    ?exe prov:qualifiedAssociation ?asso .

    OPTIONAL {
        ?exe sio:SIO_000369 ?llm_gen .
    }

    BIND(COALESCE(?llm_gen, "None") AS ?llm_gen_out)
    
    OPTIONAL {
        ?llm_out2 sio:SIO_000202 ?obj .
        ?llm_out2 sio:SIO_000232 ?llm2 .
        ?llm_gen2 prov:used ?llm2 .
    }

    BIND(COALESCE(?llm_gen2, "None") AS ?llm_gen_out2)
}

"""

def answer_template_function(
    entities:pd.DataFrame, 
    question_specs:Mapping[str, Union[str,int]], 
    sparql_specs:Mapping[str, Union[str,int]]
    ) -> GTAnswer:

    answer_template = """
    1. Executions of the {func_desc} function {decision1} a Generative Task that utilize an LLM.
    2. The {func_desc} function {decision2} by a Generative Task.

    {answer_inst}
    """
    
    _ent = entities[['flbl', 'exe', 'llm_gen_out', 'llm_gen_out2']]
    _temp_ent = format_dataframe_to_string(
        _ent,
        header_map={
            "flbl": "Function",
            'exe': "Execution",
            'llm_gen_out': "Generative Task used within execution",
            'llm_gen_out2': "Generative Task used to generate function"
        }
    )
    
    answer_nlp = regex_add_strings(
        answer_template, 
        func_desc = question_specs["func_desc"],
        answer_inst = _temp_ent,
        decision1 = "contains" if _ent['llm_gen_out'].tolist()[0] != "None" else "does not contain",
        decision2 = "is generated" if _ent['llm_gen_out2'].tolist()[0] != "None" else "is not generated"
        )
    
    return GTAnswer(
        answer_nlp= answer_nlp,
        entities= _ent.to_dict(orient="records")
    )

gts = build_gt_from_template(
        template=question_template,
        answer_template=answer_template_function,
        sparql_query=sparql_template,
        specs_template=template_spec,
        specs_sparql=sparql_spec,
        graph_manager= graph_manager,
        verbose=True
    )

for gt in gts:
    gt.qtype = ["single", "bool"]
    gt.decision = (gt.entities[0]['llm_gen_out'] != "None") and (gt.entities[0]['llm_gen_out2'] != "None")

gt_list.extend(gts)

ic| rquestion: '''
                Does any of the executions of the sparql query post processing function made has a Generative Task within as a component and also sparql query post processing 
                function itself is generated by a Generative Task?
                '''
ic| entities:                                            flbl  \
              0  Query Result Post-Processor@en^^<xsd:string>   
              
                                                               exe llm_gen_out  \
              0  http://testwebsite/testProgram#id_202604122007...        None   
              
                                                      llm_gen_out2  
              0  http://testwebsite/testProgram#Generative_Task...  
ic| answer: GTAnswer(answer_nlp='
                1. Executions of the sparql query post processing function does not contain a Generative Task that utilize an LLM.
                2. The sparql query post processing function is generated by a Generative T

ic| rquestion: '''
                Does any of the executions of the system prompt template function made has a Generative Task within as a component and also system prompt template 
                function itself is generated by a Generative Task?
                '''
ic| entities:                                                 flbl  \
              0  System Prompt Template Generator@en^^<xsd:string>   
              
                                                               exe llm_gen_out  \
              0  http://testwebsite/testProgram#id_202604122007...        None   
              
                                                      llm_gen_out2  
              0  http://testwebsite/testProgram#Generative_Task...  
ic| answer: GTAnswer(answer_nlp='
                1. Executions of the system prompt template function does not contain a Generative Task that utilize an LLM.
                2. The system prompt template function is generated by a Generative Task.
         

In [20]:
# Question MULT 10 : Does any of the executions of the function contain a Generative Task within and the function itself is generated by an LLM


template_spec = [ {
    "func_desc": "information extraction"
}, {
    "func_desc": "llm chat function"
}]

sparql_spec = [{
    "func_desc_term":"Extracts entities or relevant information"
}, {
    "func_desc_term": "Chat with the LLM"
}]

question_template = """
What are the inputs of the Generative Task that was a component in the executions of  the {func_desc} function?
"""

sparql_template = """
SELECT DISTINCT ?flbl ?exe ?lbl
WHERE {
    ?obj dc:description ?desc .
    ?obj rdfs:label ?flbl .
    FILTER(REGEX(?desc, "{func_desc_term}","i"))

    ?asso prov:hadPlan ?obj .
    ?exe prov:qualifiedAssociation ?asso .

    ?exe sio:SIO_000369 ?llm_gen .
    ?llm_gen prov:used ?llm .
    ?llm sio:SIO_000230 ?inp .
    ?inp prov:value ?lbl . 
}
"""

def answer_template_function(
    entities:pd.DataFrame, 
    question_specs:Mapping[str, Union[str,int]], 
    sparql_specs:Mapping[str, Union[str,int]]
    ) -> GTAnswer:

    answer_template = """
    The inputs to the LLM used for the Generative Task in the executions of the {func_desc} function

    {answer_inst}
    """
    
    _ent = entities[['flbl', 'exe', 'lbl']]
    _temp_ent = format_dataframe_to_string(
        _ent,
        header_map={
            "flbl": "Function",
            'exe': "Execution",
            'lbl': "Inputs for the Generative Task used within execution"
        }
    )
    
    answer_nlp = regex_add_strings(
        answer_template, 
        func_desc = question_specs["func_desc"],
        answer_inst = _temp_ent
        )
    
    return GTAnswer(
        answer_nlp= answer_nlp,
        entities= _ent[['exe', 'lbl']].to_dict(orient="records")
    )

gts = build_gt_from_template(
        template=question_template,
        answer_template=answer_template_function,
        sparql_query=sparql_template,
        specs_template=template_spec,
        specs_sparql=sparql_spec,
        graph_manager= graph_manager,
        verbose=True
    )

for gt in gts:
    gt.qtype = ["multi", "entity", 'label']
gt_list.extend(gts)

ic| rquestion: '''
                What are the inputs of the Generative Task that was a component in the executions of  the information extraction function?
                '''
ic| entities:                                      flbl  \
              0  Information Extractor@en^^<xsd:string>   
              
                                                               exe  \
              0  http://testwebsite/testProgram#id_202604122007...   
              
                                                               lbl  
              0  Before your workout, it's essential to eat foo...  
ic| answer: GTAnswer(answer_nlp="
                The inputs to the LLM used for the Generative Task in the executions of the information extraction function
            
                Function | Execution | Inputs for the Generative Task used within execution
            Information Extractor@en^^<xsd:string> | http://testwebsite/testProgram#id_20260412200742_397 | Before your workout, it's

In [21]:
# Question MULT 11 :


template_spec = [ {
    "func_desc": "information extraction"
}, {
    "func_desc": "llm chat function"
}]

sparql_spec = [{
    "func_desc_term":"Extracts entities or relevant information"
}, {
    "func_desc_term": "Chat with the LLM"
}]

question_template = """
What are the outputs of the Generative Task that was a component in the executions of the {func_desc} function?
"""

sparql_template = """
SELECT DISTINCT ?flbl ?exe ?lbl
WHERE {
    ?obj dc:description ?desc .
    ?obj rdfs:label ?flbl .
    FILTER(REGEX(?desc, "{func_desc_term}","i"))

    ?asso prov:hadPlan ?obj .
    ?exe prov:qualifiedAssociation ?asso .

    ?exe sio:SIO_000369 ?llm_gen .
    ?llm_gen prov:used ?llm .
    ?llm sio:SIO_000229 ?out .
    ?out sio:SIO_000202 ?data . 
    ?data prov:value ?lbl . 
}
"""

def answer_template_function(
    entities:pd.DataFrame, 
    question_specs:Mapping[str, Union[str,int]], 
    sparql_specs:Mapping[str, Union[str,int]]
    ) -> GTAnswer:

    answer_template = """
    The inputs to the LLM used for the Generative Task in the executions of the {func_desc} function

    {answer_inst}
    """
    
    _ent = entities[['flbl', 'exe', 'lbl']]
    _temp_ent = format_dataframe_to_string(
        _ent,
        header_map={
            "flbl": "Function",
            'exe': "Execution",
            'lbl': "Inputs for the Generative Task used within execution"
        }
    )
    
    answer_nlp = regex_add_strings(
        answer_template, 
        func_desc = question_specs["func_desc"],
        answer_inst = _temp_ent
        )
    
    return GTAnswer(
        answer_nlp= answer_nlp,
        entities= _ent.to_dict(orient="records")
    )

gts = build_gt_from_template(
        template=question_template,
        answer_template=answer_template_function,
        sparql_query=sparql_template,
        specs_template=template_spec,
        specs_sparql=sparql_spec,
        graph_manager= graph_manager,
        verbose=True
    )

for gt in gts:
    gt.qtype = ["multi", "entity", 'label']
gt_list.extend(gts)

ic| rquestion: '''
                What are the outputs of the Generative Task that was a component in the executions of the information extraction function?
                '''
ic| entities:                                       flbl  \
              0   Information Extractor@en^^<xsd:string>   
              1   Information Extractor@en^^<xsd:string>   
              2   Information Extractor@en^^<xsd:string>   
              3   Information Extractor@en^^<xsd:string>   
              4   Information Extractor@en^^<xsd:string>   
              5   Information Extractor@en^^<xsd:string>   
              6   Information Extractor@en^^<xsd:string>   
              7   Information Extractor@en^^<xsd:string>   
              8   Information Extractor@en^^<xsd:string>   
              9   Information Extractor@en^^<xsd:string>   
              10  Information Extractor@en^^<xsd:string>   
              11  Information Extractor@en^^<xsd:string>   
              12  Information Extractor@en

In [22]:
# Question MULT 11 :

_spec = [ {
    "func_desc": "information extraction",
    "func_desc_term":"Extracts entities or relevant information"
}, {
    "func_desc": "llm chat function",
    "func_desc_term": "Chat with the LLM"
}]

template_spec = []
sparql_spec = []

for _s in _spec:
    exe_query = """
    SELECT DISTINCT ?exe
    WHERE {
        ?obj dc:description ?desc .
        ?obj rdfs:label ?flbl .
        FILTER(REGEX(?desc, "{func_desc_term}","i"))
        
        ?asso prov:hadPlan ?obj .
        ?exe prov:qualifiedAssociation ?asso .      
        
        }
    """
    executions = graph_manager.query(
        regex_add_strings(exe_query, func_desc_term = _s["func_desc_term"])
    )


    for obj in executions.to_dict("records"):
        template_spec.append({
            "exe_id":obj['exe']
        })
        
        sparql_spec.append({
            "exe_id":obj['exe']
        })

question_template = """
What execution created the output data that was used as an input for the generative task within the execution {exe_id}?
"""

sparql_template = """
SELECT DISTINCT ?exe2
    WHERE {
        <{exe_id}> sio:SIO_000369 ?llm_gen .
        ?llm_gen prov:used ?llm .
        ?llm sio:SIO_000229 ?out .
        ?out sio:SIO_000202 ?data . 
        ?data prov:wasGeneratedBy ?exe2 . 
}
"""

def answer_template_function(
    entities:pd.DataFrame, 
    question_specs:Mapping[str, Union[str,int]], 
    sparql_specs:Mapping[str, Union[str,int]]
    ) -> GTAnswer:

    answer_template = """
    The inputs to the Generative Task in the execution of {exe_id} was generated by executions

    {answer_inst}
    """
    
    _ent = entities[['exe2']]
    _temp_ent = format_dataframe_to_string(
        _ent,
        header_map={
            'exe2': "Execution"
        }
    )
    
    answer_nlp = regex_add_strings(
        answer_template, 
        exe_id = question_specs["exe_id"],
        answer_inst = _temp_ent
        )
    
    return GTAnswer(
        answer_nlp= answer_nlp,
        entities= _ent.to_dict(orient="records")
    )

gts = build_gt_from_template(
        template=question_template,
        answer_template=answer_template_function,
        sparql_query=sparql_template,
        specs_template=template_spec,
        specs_sparql=sparql_spec,
        graph_manager= graph_manager,
        verbose=True
    )

for gt in gts:
    gt.qtype = ["multi", "entity", 'label']
gt_list.extend(gts)

ic| rquestion: '''
                What execution created the output data that was used as an input for the generative task within the execution http://testwebsite/testProgram#id_20260412200742_397?
                '''
ic| entities:                                                  exe2
              0   http://testwebsite/testProgram#id_202604122007...
              1   http://testwebsite/testProgram#id_202604122007...
              2   http://testwebsite/testProgram#id_202604122007...
              3   http://testwebsite/testProgram#id_202604122007...
              4   http://testwebsite/testProgram#id_202604122007...
              5   http://testwebsite/testProgram#id_202604122007...
              6   http://testwebsite/testProgram#id_202604122007...
              7   http://testwebsite/testProgram#id_202604122007...
              8   http://testwebsite/testProgram#id_202604122007...
              9   http://testwebsite/testProgram#id_202604122007...
              10  http://testwebs

In [23]:
# Question MULT 12 :

_spec = [ {
    "func_desc": "information extraction",
    "func_desc_term":"Extracts entities or relevant information"
}, {
    "func_desc": "sparql query function",
    "func_desc_term": "Builds SPARQL queries to extract"
}]

template_spec = []
sparql_spec = []

for _s in _spec:
    exe_query = """
    SELECT DISTINCT ?exe
    WHERE {
        ?obj dc:description ?desc .
        ?obj rdfs:label ?flbl .
        FILTER(REGEX(?desc, "{func_desc_term}","i"))
        
        ?asso prov:hadPlan ?obj .
        ?exe prov:qualifiedAssociation ?asso .      
        
        }
    """
    executions = graph_manager.query(
        regex_add_strings(exe_query, func_desc_term = _s["func_desc_term"])
    )


    for obj in executions.to_dict("records"):
        template_spec.append({
            "exe_id":obj['exe']
        })
        
        sparql_spec.append({
            "exe_id":obj['exe']
        })

question_template = """
What Generative Task generated the data that was used by the execution {exe_id}?
"""

sparql_template = """
SELECT DISTINCT ?gtask ?glbl
            WHERE {
                <{exe_id}> prov:used ?data .
                ?llm_out sio:SIO_000202 ?data .
                ?llm_out sio:SIO_000232 ?llm .
                ?gtask prov:used ?llm .
                ?gtask rdfs:label ?glbl
        }
"""

def answer_template_function(
    entities:pd.DataFrame, 
    question_specs:Mapping[str, Union[str,int]], 
    sparql_specs:Mapping[str, Union[str,int]]
    ) -> GTAnswer:

    answer_template = """
    The Generative Task that generated the data that was used by the execution {exe_id} is 

    {answer_inst}
    """
    
    _ent = entities[['gtask', 'glbl']]
    _temp_ent = format_dataframe_to_string(
        _ent,
        header_map={
            'gtask': "Generative Task"
        }
    )
    
    answer_nlp = regex_add_strings(
        answer_template, 
        exe_id = question_specs["exe_id"],
        answer_inst = _temp_ent
        )
    
    return GTAnswer(
        answer_nlp= answer_nlp,
        entities= _ent[['gtask']].to_dict(orient="records")
    )

gts = build_gt_from_template(
        template=question_template,
        answer_template=answer_template_function,
        sparql_query=sparql_template,
        specs_template=template_spec,
        specs_sparql=sparql_spec,
        graph_manager= graph_manager,
        verbose=True
    )

for gt in gts:
    gt.qtype = ["single", "entity"]
gt_list.extend(gts)

ic| rquestion: '''
                What Generative Task generated the data that was used by the execution http://testwebsite/testProgram#id_20260412200742_397?
                '''
ic| entities:                                                gtask  \
              0  http://testwebsite/testProgram#Generative_Task...   
              
                                                              glbl  
              0  LLM Generates an text for 
              system prompt: You...  
ic| answer: GTAnswer(answer_nlp='
                The Generative Task that generated the data that was used by the execution http://testwebsite/testProgram#id_20260412200742_397 is 
            
                Generative Task | glbl
            http://testwebsite/testProgram#Generative_Task-id_20260412200740_99 | LLM Generates an text for 
            system prompt: You are a health care assistant and you will respond to the user who is 30 years old, identifies as Male, diagnosed with Diabetic. Consider thei

ic| rquestion: '''
                What Generative Task generated the data that was used by the execution http://testwebsite/testProgram#id_20260412200743_952?
                '''
ic| entities:                                                gtask  \
              0  http://testwebsite/testProgram#Generative_Task...   
              
                                                              glbl  
              0  Extracts important entities for 
              text: Before...  
ic| answer: GTAnswer(answer_nlp="
                The Generative Task that generated the data that was used by the execution http://testwebsite/testProgram#id_20260412200743_952 is 
            
                Generative Task | glbl
            http://testwebsite/testProgram#Generative_Task-id_20260412200742_397 | Extracts important entities for 
            text: Before your workout, it's essential to eat foods that provide a quick energy boost while being mindful of your diabetes management. Since your die

In [24]:
# Input port data created by AI 

_spec = [{
    "func_desc":"sparql query post processing",
    "func_desc_term":"Post-processes the query results"
}, {
    "func_desc": "system prompt template",
    "func_desc_term":"Generates a system prompt"
}, {
    "func_desc": "information extraction",
    "func_desc_term":"Extracts entities or relevant information"
}, {
    "func_desc": "sparql query function",
    "func_desc_term": "Builds SPARQL queries to extract"
}, {
    "func_desc": "llm chat function",
    "func_desc_term": "Chat with the LLM"
}]

template_spec = []
sparql_spec = []

for _s in _spec:
    exe_query = """
    SELECT DISTINCT ?exe
    WHERE {
        ?obj dc:description ?desc .
        ?obj rdfs:label ?flbl .
        FILTER(REGEX(?desc, "{func_desc_term}","i"))
        
        ?asso prov:hadPlan ?obj .
        ?exe prov:qualifiedAssociation ?asso .      
        
        }
    """
    executions = graph_manager.query(
        regex_add_strings(exe_query, func_desc_term = _s["func_desc_term"])
    )


    for obj in executions.to_dict("records"):
        usage_query = """
        SELECT DISTINCT ?usage ?lbl
        WHERE {
            <{exe_id}> prov:qualifiedUsage ?usage .
            ?usage provone:hadInPort ?port .
            ?port rdfs:label ?lbl .
            }
        """
        usages = graph_manager.query(
                regex_add_strings(
                    usage_query,
                    exe_id = obj["exe"]
                )
            )
        
        for usage in usages.to_dict("records"):
            # df = graph_manager.query(
            #     regex_add_strings(
            #         sparql_template,
            #         exe_id = obj["exe"],
            #         port_label = usage["lbl"].split("@")[0]
            #     )
            # )

            # print(df)
            
            template_spec.append({
                "exe_id":obj['exe'],
                "port_label" : usage["lbl"].split("@")[0]
            })
            
            sparql_spec.append({
                "exe_id":obj['exe'],
                "port_label" : usage["lbl"].split("@")[0]
            })
            
question_template = """
Was the input to the {port_label} in the execution {exe_id} generated by a Generative Task?
"""

sparql_template = """
SELECT DISTINCT ?plbl ?gtask2
            WHERE {
                <{exe_id}> prov:qualifiedUsage ?usage .
                ?usage provone:hadInPort ?port .
                ?port rdfs:label ?plbl .
                FILTER(REGEX(?plbl, "{port_label}","i"))
                
                ?usage provone:hadEntity ?data .
                OPTIONAL {
                    ?llm_out sio:SIO_000202 ?data .
                    ?llm_out sio:SIO_000232 ?llm .
                    ?gtask prov:used ?llm .
                }
                
                BIND(COALESCE(?gtask, "None") AS ?gtask2)
        }
"""

def answer_template_function(
    entities:pd.DataFrame, 
    question_specs:Mapping[str, Union[str,int]], 
    sparql_specs:Mapping[str, Union[str,int]]
    ) -> GTAnswer:

    answer_template = """
    The input to the {port_label} in the execution {exe_id} {decision} generated by a 
    Generative Task

    {answer_inst}
    """
    
    _ent = entities[['gtask2']]
    _temp_ent = format_dataframe_to_string(
        _ent,
        header_map={
            'gtask2': "Generative Task"
        }
    )
    
    answer_nlp = regex_add_strings(
        answer_template, 
        exe_id = question_specs["exe_id"],
        port_label = question_specs["port_label"],
        decision = "was" if _ent['gtask2'].tolist()[0] != "None" else "was not",
        answer_inst = _temp_ent
        )
    
    return GTAnswer(
        answer_nlp= answer_nlp,
        entities= _ent.to_dict(orient="records")
    )

gts = build_gt_from_template(
        template=question_template,
        answer_template=answer_template_function,
        sparql_query=sparql_template,
        specs_template=template_spec,
        specs_sparql=sparql_spec,
        graph_manager= graph_manager,
        verbose=True
    )

for gt in gts:
    gt.qtype = ["single", "bool"]
gt_list.extend(gts)

ic| rquestion: '''
                Was the input to the Post-processing input port: kg_knowledge in the execution http://testwebsite/testProgram#id_20260412200745_507 generated by a Generative Task?
                '''
ic| entities:                                                 plbl gtask2
              0  Post-processing input port: kg_knowledge@en^^<...   None
ic| answer: GTAnswer(answer_nlp='
                The input to the Post-processing input port: kg_knowledge in the execution http://testwebsite/testProgram#id_20260412200745_507 was not generated by a 
                Generative Task
            
                Generative Task
            None
                ', entities=[{'gtask2': 'None'}])
ic| rquestion: '''
                Was the input to the Input port for gender in the execution http://testwebsite/testProgram#id_20260412200734_778 generated by a Generative Task?
                '''
ic| entities:                                      plbl gtask2
              0  Input p

ic| answer: GTAnswer(answer_nlp='
                The input to the Input port for gender in the execution http://testwebsite/testProgram#id_20260412200734_778 was not generated by a 
                Generative Task
            
                Generative Task
            None
                ', entities=[{'gtask2': 'None'}])
ic| rquestion: '''
                Was the input to the Input port for height in the execution http://testwebsite/testProgram#id_20260412200734_778 generated by a Generative Task?
                '''
ic| entities:                                      plbl gtask2
              0  Input port for height@en^^<xsd:string>   None
ic| answer: GTAnswer(answer_nlp='
                The input to the Input port for height in the execution http://testwebsite/testProgram#id_20260412200734_778 was not generated by a 
                Generative Task
            
                Generative Task
            None
                ', entities=[{'gtask2': 'None'}])
ic| rquestion: '''
 

In [25]:
# Input port data created by AI 

_spec = [{
    "func_desc":"sparql query post processing",
    "func_desc_term":"Post-processes the query results"
}, {
    "func_desc": "system prompt template",
    "func_desc_term":"Generates a system prompt"
}, {
    "func_desc": "information extraction",
    "func_desc_term":"Extracts entities or relevant information"
}, {
    "func_desc": "sparql query function",
    "func_desc_term": "Builds SPARQL queries to extract"
}, {
    "func_desc": "llm chat function",
    "func_desc_term": "Chat with the LLM"
}]

sparql_template = """
SELECT DISTINCT ?plbl ?gtask2
            WHERE {
                <{exe_id}> prov:qualifiedUsage ?usage .
                ?usage provone:hadInPort ?port .
                ?port rdfs:label ?plbl .
                FILTER(REGEX(?plbl, "{port_label}","i"))
                
                ?usage provone:hadEntity ?data .
                OPTIONAL {
                    ?llm_out sio:SIO_000202 ?data .
                    ?llm_out sio:SIO_000232 ?llm .
                    ?gtask prov:used ?llm .
                }
                
                BIND(COALESCE(?gtask, "None") AS ?gtask2)
        }
"""

template_spec = []
sparql_spec = []

for _s in _spec:
    exe_query = """
    SELECT DISTINCT ?exe
    WHERE {
        ?obj dc:description ?desc .
        ?obj rdfs:label ?flbl .
        FILTER(REGEX(?desc, "{func_desc_term}","i"))
        
        ?asso prov:hadPlan ?obj .
        ?exe prov:qualifiedAssociation ?asso .      
        
        }
    """
    executions = graph_manager.query(
        regex_add_strings(exe_query, func_desc_term = _s["func_desc_term"])
    )


    for obj in executions.to_dict("records"):
        usage_query = """
        SELECT DISTINCT ?usage ?lbl
        WHERE {
            <{exe_id}> prov:qualifiedUsage ?usage .
            ?usage provone:hadInPort ?port .
            ?port rdfs:label ?lbl .
            }
        """
        usages = graph_manager.query(
                regex_add_strings(
                    usage_query,
                    exe_id = obj["exe"]
                )
            )
        
        for usage in usages.to_dict("records"):
            df = graph_manager.query(
                regex_add_strings(
                    sparql_template,
                    exe_id = obj["exe"],
                    port_label = usage["lbl"].split("@")[0]
                )
            )

            print(df)
            
            template_spec.append({
                "exe_id":obj['exe'],
                "port_label" : usage["lbl"].split("@")[0]
            })
            
            sparql_spec.append({
                "exe_id":obj['exe'],
                "port_label" : usage["lbl"].split("@")[0]
            })

                                                plbl gtask2
0  Post-processing input port: kg_knowledge@en^^<...   None
                                     plbl gtask2
0  Input port for gender@en^^<xsd:string>   None
                                     plbl gtask2
0  Input port for height@en^^<xsd:string>   None
                                        plbl gtask2
0  Input port for allergies@en^^<xsd:string>   None
                                              plbl gtask2
0  Input port for foodPreferences@en^^<xsd:string>   None
                                            plbl gtask2
0  Input port for bodyMassIndex@en^^<xsd:string>   None
                                  plbl gtask2
0  Input port for age@en^^<xsd:string>   None
                                     plbl gtask2
0  Input port for weight@en^^<xsd:string>   None
                                        plbl gtask2
0  Input port for diagnosis@en^^<xsd:string>   None
                                   plbl  \
0  Input port f

In [26]:
# execution + port + was data 

In [27]:
# Question MULT 13 : Does a specific execution contain a Generative Task 

In [28]:
# what are the output data of the generative task that was used within an execution task 

In [29]:
# what are the  input data used by generative tasks that were used within an execution task

In [30]:
# 

In [31]:
os.makedirs(os.path.dirname(config.gt.save_loc), exist_ok=True)
for i, gt in enumerate(gt_list):
    gt.id = f"gt_{i}"

# common_utils.serialization.save_json(
#     {gt.id: gt.model_dump() for gt in gt_list}, config.gt.save_loc
# )



In [32]:
gt_list

[GT(id='gt_0', question='\nHow many unique "experiment execution" are there in this?\n', answer='The answer to the question is 1 unique executions.', entities=[{'obj_count': '1'}], sparql_querys=[SPARQLTemplate(template='\nSELECT (count(distinct ?ids) AS ?obj_count)\nWHERE {\n     ?obj a provone:Execution ;\n          dcterms:identifier ?ids .\n}\n', description='This SPARQL query counts the number of unique executions by counting distinct identifiers in the provone:Execution class.', inputs=None)], qtype=['single', 'numeric'], decision=None, tags={}),
 GT(id='gt_1', question='\nIn what places do we utilize AI in this workflow?\n', answer="\nThe ChatBS System utilizes LLMs to generate the following functions\n\n1. Information extraction function\n2. System Prompt template function\n3. Sparql result post processing function\n\nand further it's used within functions to\n1. Generate the LLM outputs,\n2. Used to extract key information from the system.\n", entities=[{'obj': 'ChatBS-NexGen: